# TriCheck-LK - Cross-Lingual Semantic Model

This notebook develops the multilingual semantic analysis component of TriCheck-LK.

The objective is to represent English, Sinhala and Tamil document chunks
in a shared semantic vector space and identify semantically corresponding
content across the three language versions.

Main steps:
- Load the preprocessed multilingual chunk dataset
- Load a pretrained multilingual embedding model
- Generate multilingual sentence embeddings
- Measure cosine similarity
- Perform cross-lingual chunk alignment
- Generate semantic consistency scores

In [2]:
%pip install -U sentence-transformers

  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.29.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.32.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached sentence_transformers-6.0.0-py3-no


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

from sentence_transformers import (
    SentenceTransformer,
    util
)

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
data_file = (
    "../data/processed/"
    "multilingual_chunks.jsonl"
)

chunk_df = pd.read_json(
    data_file,
    lines=True
)

print(
    "Dataset shape:",
    chunk_df.shape
)

print(
    "\nLanguages:"
)

print(
    chunk_df["language"]
    .value_counts()
)

Dataset shape: (34316, 8)

Languages:
language
Tamil      11813
English    11426
Sinhala    11077
Name: count, dtype: int64


In [5]:
model_name = (
    "intfloat/"
    "multilingual-e5-small"
)

model = SentenceTransformer(
    model_name
)

print(
    "Model loaded:",
    model_name
)

print(
    "Maximum sequence length:",
    model.max_seq_length
)

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11851.34it/s]


Model loaded: intfloat/multilingual-e5-small
Maximum sequence length: 512


In [6]:
test_texts = [
    "query: Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service under the Merit Stream - 2026",

    "query: කුසලතා ධාරාව යටතේ ශ්‍රී ලංකා පරිපාලන සේවයේ III ශ්‍රේණියට බඳවා ගැනීම සඳහා තරග විභාගය - 2026",

    "query: திறமை அடிப்படையில் இலங்கை நிர்வாக சேவையின் தரம் III-க்கு ஆட்சேர்ப்பு செய்வதற்கான போட்டிப் பரீட்சை - 2026"
]

In [7]:
test_embeddings = model.encode(
    test_texts,
    normalize_embeddings=True
)

print(
    "Embedding shape:",
    test_embeddings.shape
)

Embedding shape: (3, 384)


In [8]:
similarity_matrix = util.cos_sim(
    test_embeddings,
    test_embeddings
)

print(similarity_matrix)

tensor([[1.0000, 0.9213, 0.9273],
        [0.9213, 1.0000, 0.9742],
        [0.9273, 0.9742, 1.0000]])


In [9]:
# Select one circular that has all three languages
sample_circular_id = (
    chunk_df["circular_id"]
    .iloc[0]
)

sample_circular = (
    chunk_df[
        chunk_df["circular_id"]
        == sample_circular_id
    ]
    .copy()
)

print(
    "Circular ID:",
    sample_circular_id
)

print(
    sample_circular[
        [
            "language",
            "chunk_index",
            "extraction_status"
        ]
    ].head(20)
)

Circular ID: 2221
   language  chunk_index extraction_status
0   English            0       DIRECT_TEXT
1   English            1       DIRECT_TEXT
2   English            2       DIRECT_TEXT
3   English            3       DIRECT_TEXT
4   English            4       DIRECT_TEXT
5   English            5       DIRECT_TEXT
6   English            6       DIRECT_TEXT
7   English            7       DIRECT_TEXT
8   English            8       DIRECT_TEXT
9   English            9       DIRECT_TEXT
10  English           10       DIRECT_TEXT
11  English           11       DIRECT_TEXT
12  English           12       DIRECT_TEXT
13  English           13       DIRECT_TEXT
14  English           14       DIRECT_TEXT
15  English           15       DIRECT_TEXT
16  English           16       DIRECT_TEXT
17  English           17       DIRECT_TEXT
18  English           18       DIRECT_TEXT
19  English           19       DIRECT_TEXT


In [14]:
# Get English chunks from the selected circular

english_chunks = (
    sample_circular[
        sample_circular["language"] == "English"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)

print(
    "English chunks:",
    len(english_chunks)
)

English chunks: 25


In [15]:
english_reference = (
    english_chunks
    .iloc[0]["chunk_text"]
)

print(
    english_reference[:500]
)

Public Administration Circular : 17/2026 My number : SLAS/R/Merit/2026 Ministry of Public Administration, Provincial Councils and Local Government Independence Square Colombo 07. 10.08.2026 Secretaries of Ministries Chief Secretaries of Provinces Secretaries of Commissions District Secretaries / Government Agents Heads of Departments. Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service under the Merit Stream -2026 Applications are hereby called from quali


In [10]:
print(
    sample_circular[
        "language"
    ].value_counts()
)

language
Tamil      28
English    25
Sinhala    25
Name: count, dtype: int64


In [11]:
#cross lingual matching test
# Get English chunk 0
english_chunk = (
    sample_circular[
        (sample_circular["language"] == "English")
        & (sample_circular["chunk_index"] == 0)
    ]
    .iloc[0]["chunk_text"]
)


# Get all Sinhala chunks
sinhala_chunks = (
    sample_circular[
        sample_circular["language"] == "Sinhala"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)


# Get all Tamil chunks
tamil_chunks = (
    sample_circular[
        sample_circular["language"] == "Tamil"
    ]
    .sort_values("chunk_index")
    .reset_index(drop=True)
)


# Generate English embedding
english_embedding = model.encode(
    ["query: " + english_chunk],
    normalize_embeddings=True
)


# Generate Sinhala embeddings
sinhala_embeddings = model.encode(
    [
        "query: " + text
        for text in sinhala_chunks["chunk_text"]
    ],
    normalize_embeddings=True
)


# Generate Tamil embeddings
tamil_embeddings = model.encode(
    [
        "query: " + text
        for text in tamil_chunks["chunk_text"]
    ],
    normalize_embeddings=True
)


# Calculate similarity
sinhala_scores = util.cos_sim(
    english_embedding,
    sinhala_embeddings
)[0]

tamil_scores = util.cos_sim(
    english_embedding,
    tamil_embeddings
)[0]


# Find best matches
best_sinhala_position = (
    sinhala_scores.argmax().item()
)

best_tamil_position = (
    tamil_scores.argmax().item()
)


best_sinhala = (
    sinhala_chunks.iloc[
        best_sinhala_position
    ]
)

best_tamil = (
    tamil_chunks.iloc[
        best_tamil_position
    ]
)


print("ENGLISH CHUNK 0")
print("-" * 70)
print(english_chunk[:500])


print("\nBEST SINHALA MATCH")
print("-" * 70)

print(
    "Chunk index:",
    best_sinhala["chunk_index"]
)

print(
    "Similarity:",
    round(
        sinhala_scores[
            best_sinhala_position
        ].item(),
        4
    )
)

print(
    best_sinhala["chunk_text"][:500]
)


print("\nBEST TAMIL MATCH")
print("-" * 70)

print(
    "Chunk index:",
    best_tamil["chunk_index"]
)

print(
    "Similarity:",
    round(
        tamil_scores[
            best_tamil_position
        ].item(),
        4
    )
)

print(
    best_tamil["chunk_text"][:500]
)

ENGLISH CHUNK 0
----------------------------------------------------------------------
Public Administration Circular : 17/2026 My number : SLAS/R/Merit/2026 Ministry of Public Administration, Provincial Councils and Local Government Independence Square Colombo 07. 10.08.2026 Secretaries of Ministries Chief Secretaries of Provinces Secretaries of Commissions District Secretaries / Government Agents Heads of Departments. Competitive Examination for Recruitment to Grade III of the Sri Lanka Administrative Service under the Merit Stream -2026 Applications are hereby called from quali

BEST SINHALA MATCH
----------------------------------------------------------------------
Chunk index: 0
Similarity: 0.9001
රාජ්‍ය පරිපාලන චක්‍රලේඛ 17/2026 මලේේ අංකය SLAS/R/Merit/2026 රාජ්‍ය පරිපාලන, පළාත් සභා සහ පළාත් පාලන අමාත්යාංශය නිදහස් චතුරශ්‍රය ලේකොළඹ 07 2026.08.10 අමාත්යාංශ ලේඛකම්වරුන් පළාත් ප්‍ර ධාන ලේඛකම්වරුන් ලේකොමිෂන් සභා ලේඛකම්වරුන් දිස්ත්‍රික් ල ලේඛකම්වරුන් දිසාපතිවරුන් ලේදපාර්ත්ලේම්න්තු ප්‍ර ධ

In [12]:
tamil_url = (
    sample_circular[
        sample_circular["language"] == "Tamil"
    ]
    .iloc[0]["pdf_url"]
)

print(tamil_url)

https://pubad.gov.lk/web/images/circulars/2026/T/1786357117-17-2026-t.pdf


In [13]:
with open(
    "../data/processed/tamil_17_2026_direct_test.txt",
    encoding="utf-8"
) as file:
    tamil_direct = file.read()


with open(
    "../data/processed/tamil_17_2026_ocr_test.txt",
    encoding="utf-8"
) as file:
    tamil_ocr = file.read()

In [ ]:
english_reference = (
    english_chunks
    .iloc[0]["chunk_text"]
)

In [16]:
quality_test_texts = [
    "query: " + english_reference,
    "query: " + tamil_direct[:1000],
    "query: " + tamil_ocr[:1000]
]

quality_embeddings = model.encode(
    quality_test_texts,
    normalize_embeddings=True
)

quality_scores = util.cos_sim(
    quality_embeddings,
    quality_embeddings
)

print(
    "English ↔ Direct Tamil:",
    round(
        quality_scores[0, 1].item(),
        4
    )
)

print(
    "English ↔ OCR Tamil:",
    round(
        quality_scores[0, 2].item(),
        4
    )
)

English ↔ Direct Tamil: 0.8747
English ↔ OCR Tamil: 0.8938


In [18]:
# Generate embeddings for ALL English chunks

english_embeddings = model.encode(
    [
        "query: " + text
        for text in english_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


# Generate embeddings for ALL Sinhala chunks

sinhala_embeddings = model.encode(
    [
        "query: " + text
        for text in sinhala_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


# Generate embeddings for ALL Tamil chunks

tamil_embeddings = model.encode(
    [
        "query: " + text
        for text in tamil_chunks["chunk_text"]
    ],
    normalize_embeddings=True,
    convert_to_tensor=True
)


print(
    "English embeddings:",
    english_embeddings.shape
)

print(
    "Sinhala embeddings:",
    sinhala_embeddings.shape
)

print(
    "Tamil embeddings:",
    tamil_embeddings.shape
)

English embeddings: torch.Size([25, 384])
Sinhala embeddings: torch.Size([25, 384])
Tamil embeddings: torch.Size([28, 384])


In [19]:
en_si_similarity = util.cos_sim(
    english_embeddings,
    sinhala_embeddings
)

en_ta_similarity = util.cos_sim(
    english_embeddings,
    tamil_embeddings
)

print(
    "English-Sinhala matrix:",
    en_si_similarity.shape
)

print(
    "English-Tamil matrix:",
    en_ta_similarity.shape
)

English-Sinhala matrix: torch.Size([25, 25])
English-Tamil matrix: torch.Size([25, 28])


In [ ]:
alignment_results = []

for en_position in range(
    len(english_chunks)
):

    # Best Sinhala match
    
    best_si_position = (
        en_si_similarity[
            en_position
        ]
        .argmax()
        .item()
    )

    best_si_score = (
        en_si_similarity[
            en_position,
            best_si_position
        ]
        .item()
    )

    # Best Tamil match
    
    best_ta_position = (
        en_ta_similarity[
            en_position
        ]
        .argmax()
        .item()
    )

    best_ta_score = (
        en_ta_similarity[
            en_position,
            best_ta_position
        ]
        .item()
    )

    # Store result
    
    alignment_results.append(
        {
            "english_chunk": (
                english_chunks.iloc[
                    en_position
                ]["chunk_index"]
            ),

            "sinhala_match": (
                sinhala_chunks.iloc[
                    best_si_position
                ]["chunk_index"]
            ),

            "sinhala_score": round(
                best_si_score,
                4
            ),

            "tamil_match": (
                tamil_chunks.iloc[
                    best_ta_position
                ]["chunk_index"]
            ),

            "tamil_score": round(
                best_ta_score,
                4
            )
        }
    )


alignment_df = pd.DataFrame(
    alignment_results
)

alignment_df.head(10)

,english_chunk,sinhala_match,sinhala_score,tamil_match,tamil_score
0,0,0,0.9001,0,0.8863
1,1,1,0.8834,2,0.8503
2,2,1,0.8790,2,0.8702
3,3,3,0.8980,3,0.8875
4,4,4,0.8852,4,0.8773
5,5,5,0.8788,5,0.8703
6,6,6,0.9065,6,0.8719
7,7,1,0.8633,7,0.8594
8,8,11,0.8570,2,0.8460
9,9,8,0.8721,9,0.8725


In [ ]:
def dtw_align(similarity_matrix):

    # Convert PyTorch tensor to NumPy
    similarity = (
        similarity_matrix
        .detach()
        .cpu()
        .numpy()
    )

    n, m = similarity.shape

    # Large initial cost
    cost = np.full(
        (n + 1, m + 1),
        np.inf
    )

    cost[0, 0] = 0


    # Store movement direction
    backtrack = np.zeros(
        (n + 1, m + 1),
        dtype=int
    )


    for i in range(1, n + 1):

        for j in range(1, m + 1):

            # High similarity = low cost
            local_cost = (
                1
                - similarity[
                    i - 1,
                    j - 1
                ]
            )

            previous_costs = [
                cost[i - 1, j - 1],  # diagonal
                cost[i - 1, j],      # vertical
                cost[i, j - 1]       # horizontal
            ]

            best_move = np.argmin(
                previous_costs
            )

            cost[i, j] = (
                local_cost
                + previous_costs[
                    best_move
                ]
            )

            backtrack[i, j] = (
                best_move
            )

    # Recover alignment path

    i = n
    j = m

    path = []

    while i > 0 and j > 0:

        path.append(
            (
                i - 1,
                j - 1
            )
        )

        move = backtrack[i, j]

        if move == 0:
            i -= 1
            j -= 1

        elif move == 1:
            i -= 1

        else:
            j -= 1


    path.reverse()

    return path

#English ↔ Sinhala and English ↔ Tamil

In [22]:
en_si_path = dtw_align(
    en_si_similarity
)

en_ta_path = dtw_align(
    en_ta_similarity
)

print(
    "EN-SI alignment path length:",
    len(en_si_path)
)

print(
    "EN-TA alignment path length:",
    len(en_ta_path)
)

EN-SI alignment path length: 26
EN-TA alignment path length: 28


In [23]:
print("English ↔ Sinhala")

for en_pos, si_pos in en_si_path[:15]:

    score = (
        en_si_similarity[
            en_pos,
            si_pos
        ]
        .item()
    )

    print(
        f"EN {en_pos} → SI {si_pos}"
        f" | {score:.4f}"
    )


print("\nEnglish ↔ Tamil")

for en_pos, ta_pos in en_ta_path[:15]:

    score = (
        en_ta_similarity[
            en_pos,
            ta_pos
        ]
        .item()
    )

    print(
        f"EN {en_pos} → TA {ta_pos}"
        f" | {score:.4f}"
    )

English ↔ Sinhala
EN 0 → SI 0 | 0.9001
EN 1 → SI 1 | 0.8834
EN 2 → SI 2 | 0.8402
EN 3 → SI 3 | 0.8980
EN 4 → SI 4 | 0.8852
EN 5 → SI 5 | 0.8788
EN 6 → SI 6 | 0.9065
EN 7 → SI 6 | 0.8629
EN 8 → SI 7 | 0.8405
EN 9 → SI 8 | 0.8721
EN 10 → SI 9 | 0.8649
EN 11 → SI 10 | 0.8716
EN 12 → SI 11 | 0.8637
EN 13 → SI 12 | 0.8846
EN 14 → SI 13 | 0.8587

English ↔ Tamil
EN 0 → TA 0 | 0.8863
EN 1 → TA 1 | 0.8362
EN 2 → TA 2 | 0.8702
EN 3 → TA 3 | 0.8875
EN 4 → TA 4 | 0.8773
EN 5 → TA 5 | 0.8703
EN 6 → TA 6 | 0.8719
EN 7 → TA 7 | 0.8594
EN 8 → TA 8 | 0.8394
EN 9 → TA 9 | 0.8725
EN 10 → TA 10 | 0.8754
EN 10 → TA 11 | 0.8586
EN 11 → TA 12 | 0.8503
EN 12 → TA 13 | 0.8466
EN 13 → TA 14 | 0.8369


#clean DTW alignment table

In [24]:
def build_alignment_table(
    path,
    similarity_matrix,
    source_chunks,
    target_chunks,
    target_language
):

    results = []

    for source_pos, target_pos in path:

        score = (
            similarity_matrix[
                source_pos,
                target_pos
            ]
            .item()
        )

        results.append(
            {
                "english_chunk": (
                    source_chunks
                    .iloc[source_pos]["chunk_index"]
                ),

                f"{target_language.lower()}_chunk": (
                    target_chunks
                    .iloc[target_pos]["chunk_index"]
                ),

                "similarity": round(
                    score,
                    4
                )
            }
        )

    return pd.DataFrame(results)

In [25]:
en_si_alignment = build_alignment_table(
    en_si_path,
    en_si_similarity,
    english_chunks,
    sinhala_chunks,
    "Sinhala"
)

en_ta_alignment = build_alignment_table(
    en_ta_path,
    en_ta_similarity,
    english_chunks,
    tamil_chunks,
    "Tamil"
)

#summary statistics

In [26]:
print("English ↔ Sinhala")
print(
    en_si_alignment["similarity"]
    .describe()
)

print("\nEnglish ↔ Tamil")
print(
    en_ta_alignment["similarity"]
    .describe()
)

English ↔ Sinhala
count    26.000000
mean      0.874650
std       0.019402
min       0.840200
25%       0.864000
50%       0.872700
75%       0.885050
max       0.911900
Name: similarity, dtype: float64

English ↔ Tamil
count    28.000000
mean      0.860439
std       0.019228
min       0.834400
25%       0.843450
50%       0.859000
75%       0.873225
max       0.897700
Name: similarity, dtype: float64


In [27]:
print(
    "\nLowest EN-SI matches:"
)

display(
    en_si_alignment
    .nsmallest(
        5,
        "similarity"
    )
)


print(
    "\nLowest EN-TA matches:"
)

display(
    en_ta_alignment
    .nsmallest(
        5,
        "similarity"
    )
)


Lowest EN-SI matches:


,english_chunk,sinhala_chunk,similarity
2,2,2,0.8402
19,19,18,0.8402
8,8,7,0.8405
21,21,20,0.8563
14,14,13,0.8587



Lowest EN-TA matches:


,english_chunk,tamil_chunk,similarity
18,17,18,0.8344
1,1,1,0.8362
14,13,14,0.8369
19,18,19,0.8393
8,8,8,0.8394


#create baseline score summary

In [28]:
sample_summary = {
    "circular_id": sample_circular_id,

    "en_si_mean": (
        en_si_alignment["similarity"]
        .mean()
    ),

    "en_si_min": (
        en_si_alignment["similarity"]
        .min()
    ),

    "en_ta_mean": (
        en_ta_alignment["similarity"]
        .mean()
    ),

    "en_ta_min": (
        en_ta_alignment["similarity"]
        .min()
    )
}

sample_summary

{'circular_id': np.int64(2221),
 'en_si_mean': np.float64(0.8746499999999998),
 'en_si_min': np.float64(0.8402),
 'en_ta_mean': np.float64(0.8604392857142857),
 'en_ta_min': np.float64(0.8344)}

In [4]:
import pandas as pd

chunks_df = pd.read_json(
    "../data/processed/multilingual_chunks.jsonl",
    lines=True
)

sample_chunks = chunks_df[
    (chunks_df["circular_id"] == 2221)
    &
    (chunks_df["language"] == "English")
].sort_values("chunk_index")

print("Saved training chunks:", len(sample_chunks))

print(
    sample_chunks[
        ["chunk_index", "chunk_text"]
    ]
    .assign(
        length=lambda df: df["chunk_text"].str.len()
    )[
        ["chunk_index", "length"]
    ]
)

Saved training chunks: 25
    chunk_index  length
0             0     867
1             1     977
2             2     789
3             3     895
4             4     984
5             5    1000
6             6     982
7             7     723
8             8     831
9             9     976
10           10     833
11           11     972
12           12     946
13           13     943
14           14     900
15           15     884
16           16     937
17           17     854
18           18     971
19           19     942
20           20    1000
21           21     998
22           22     807
23           23     844
24           24     741


In [3]:
print(chunks_df.columns.tolist())

['circular_id', 'circular_number', 'year', 'language', 'pdf_url', 'extraction_status', 'chunk_text', 'chunk_index']
